# 🚨 Global Health Security: Bundibugyo Ebola Outbreak Surveillance & Cross-Border Risk Mitigation Engine

**Epidemiological Simulation & Risk Scoring System** | WHO PHEIC Framework | 180-Day Transmission Dynamics

---

### Regulatory Framework
- **WHO PHEIC Classification**: Public Health Emergency of International Concern (2026)
- **International Health Regulations (IHR 2005)**: Articles 6–8 (Risk Assessment & Cross-Border Notification)
- **AFRO Regional Emergency Response**: DRC–Uganda Border Coordination Protocol
- **Methodology**: Adapted SEIR Model + Multi-Factor Risk Scoring + Logistics Constraint Modeling

*This notebook simulates transmission velocity, contact tracing effectiveness, cross-border screening, and resource constraints in a realistic 180-day outbreak scenario across 6 health zones in Ituri Province, DRC, with 4 Points of Entry (PoEs) to Uganda.*


# Environment setup

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
from scipy import stats

In [5]:
plt.rcParams['figure.dpi'] = 130
plt.rcParams['font.family'] = 'DejaVu Sans'
sns.set_theme(style='whitegrid')

RISK_COLORS = {'Low':'#3BAB6F', 'Medium':'#F39C12', 'High':'#E8834D', 'Critical':'#E74C3C'}
PALETTE = ['#2D6A9F','#E74C3C','#3BAB6F','#F39C12','#9B59B6','#1ABC9C','#E8834D','#34495E']

SIMULATION_DAYS = 180
POPULATION = 200_000
NUM_HEALTH_ZONES = 6
NUM_POES = 4
NUM_CONTACTS_PER_CASE = 4

print('✓ Environment initialized')
print(f'✓ Simulation: {SIMULATION_DAYS} days | Population: {POPULATION:,}')

✓ Environment initialized
✓ Simulation: 180 days | Population: 200,000


### Generation of Data

In [6]:
health_zones = {
    'Mongbwalu': {'population': 45000, 'mining_intensity': 0.95, 'urban_density': 0.3, 'healthcare_tier': 'District'},
    'Pinga': {'population': 35000, 'mining_intensity': 0.85, 'urban_density': 0.2, 'healthcare_tier': 'Health Center'},
    'Bunia': {'population': 55000, 'mining_intensity': 0.2, 'urban_density': 0.7, 'healthcare_tier': 'Regional'},
    'Nyankunde': {'population': 20000, 'mining_intensity': 0.4, 'urban_density': 0.4, 'healthcare_tier': 'Health Center'},
    'Komanda': {'population': 25000, 'mining_intensity': 0.6, 'urban_density': 0.15, 'healthcare_tier': 'Health Center'},
    'Aru': {'population': 20000, 'mining_intensity': 0.3, 'urban_density': 0.25, 'healthcare_tier': 'Health Center'}
}

zones_df = pd.DataFrame.from_dict(health_zones, orient='index')
zones_df['zone_name'] = zones_df.index
zones_df = zones_df.reset_index(drop=True)

print('Health Zones Initialized:')
print(zones_df[['zone_name', 'population', 'mining_intensity', 'healthcare_tier']])

poes = {
    'Arua Border': {'daily_traffic': 450, 'thermal_capacity': 80, 'zone': 'Aru'},
    'Pakwach Crossing': {'daily_traffic': 380, 'thermal_capacity': 60, 'zone': 'Komanda'},
    'Mahagi Port': {'daily_traffic': 520, 'thermal_capacity': 100, 'zone': 'Pinga'},
    'Bunia Air/Road Hub': {'daily_traffic': 650, 'thermal_capacity': 120, 'zone': 'Bunia'}
}

poes_df = pd.DataFrame.from_dict(poes, orient='index')
poes_df['poe_name'] = poes_df.index
poes_df = poes_df.reset_index(drop=True)

print('\nPoints of Entry (PoEs):')
print(poes_df[['poe_name', 'daily_traffic', 'thermal_capacity', 'zone']])

Health Zones Initialized:
   zone_name  population  mining_intensity healthcare_tier
0  Mongbwalu       45000              0.95        District
1      Pinga       35000              0.85   Health Center
2      Bunia       55000              0.20        Regional
3  Nyankunde       20000              0.40   Health Center
4    Komanda       25000              0.60   Health Center
5        Aru       20000              0.30   Health Center

Points of Entry (PoEs):
             poe_name  daily_traffic  thermal_capacity     zone
0         Arua Border            450                80      Aru
1    Pakwach Crossing            380                60  Komanda
2         Mahagi Port            520               100    Pinga
3  Bunia Air/Road Hub            650               120    Bunia


### Setting up and Running of simulation

In [7]:
class SEIRSimulator:
    def __init__(self, population, zones_df, mining_boost=True):
        self.population = population
        self.zones_df = zones_df
        self.beta_base = 0.35
        self.sigma = 1/5.1
        self.gamma = 1/10
        self.cfr_healthcare = 0.25
        self.cfr_community = 0.88
        self.contact_rate_base = 4
        
    def get_transmission_rate(self, zone_name):
        zone_data = self.zones_df[self.zones_df['zone_name'] == zone_name]
        if zone_data.empty:
            return self.beta_base
        mining_int = zone_data['mining_intensity'].values[0]
        boost = 1 + (mining_int * 0.45)
        return self.beta_base * boost
    
    def simulate(self):
        days = SIMULATION_DAYS
        results = []
        
        zone_states = {}
        for idx, row in self.zones_df.iterrows():
            zone = row['zone_name']
            zone_pop = row['population']
            seed_cases = 5 if zone == 'Mongbwalu' else 0
            zone_states[zone] = {
                'S': zone_pop - seed_cases, 'E': seed_cases, 'I': 0, 'R': 0, 'D': 0,
                'Confirmed': 0, 'Suspected': 0, 'HCW_Deaths': 0, 'Comm_Deaths': 0
            }
        
        for day in range(days):
            for zone in zone_states:
                state = zone_states[zone]
                zone_data = self.zones_df[self.zones_df['zone_name'] == zone].iloc[0]
                zone_pop = zone_data['population']
                beta = self.get_transmission_rate(zone)
                N = zone_pop
                S, E, I, R, D = state['S'], state['E'], state['I'], state['R'], state['D']
                
                contact_rate = self.contact_rate_base * (1 + zone_data['urban_density'] * 0.3)
                foi = (beta * I / N) * contact_rate
                new_E = np.random.binomial(S, min(foi, 0.5))
                new_I = np.random.binomial(E, self.sigma)
                
                hcw_prop = 0.15 if I > 0 else 0
                community_prop = 1 - hcw_prop
                hcw_deaths = np.random.binomial(int(I * hcw_prop), self.cfr_healthcare)
                comm_deaths = np.random.binomial(int(I * community_prop), self.cfr_community)
                new_D = hcw_deaths + comm_deaths
                new_R = np.random.binomial(max(I - new_D, 0), self.gamma)
                
                state['S'] = max(S - new_E, 0)
                state['E'] = max(E + new_E - new_I, 0)
                state['I'] = max(I + new_I - new_R - new_D, 0)
                state['R'] = R + new_R
                state['D'] = D + new_D
                
                new_confirmed = int(new_R * 0.7)
                state['Confirmed'] += new_confirmed
                state['Suspected'] = max(state['E'] + state['I'], 0)
                state['HCW_Deaths'] += hcw_deaths
                state['Comm_Deaths'] += comm_deaths
            
            daily_agg = {'Day': day + 1, 'Date': datetime(2026, 1, 1) + timedelta(days=day)}
            for zone in zone_states:
                for key in ['S', 'E', 'I', 'R', 'D', 'Confirmed', 'Suspected', 'HCW_Deaths', 'Comm_Deaths']:
                    col_name = f'{zone}_{key}'
                    daily_agg[col_name] = zone_states[zone][key]
            results.append(daily_agg)
        
        return pd.DataFrame(results), zone_states

print('Running 180-day SEIR simulation...')
simulator = SEIRSimulator(POPULATION, zones_df, mining_boost=True)
seir_df, final_zone_states = simulator.simulate()

seir_df['Total_Confirmed'] = seir_df[[col for col in seir_df.columns if col.endswith('_Confirmed')]].sum(axis=1)
seir_df['Total_Suspected'] = seir_df[[col for col in seir_df.columns if col.endswith('_Suspected')]].sum(axis=1)
seir_df['Total_Deaths'] = seir_df[[col for col in seir_df.columns if col.endswith('_D')]].sum(axis=1)
seir_df['Total_HCW_Deaths'] = seir_df[[col for col in seir_df.columns if col.endswith('_HCW_Deaths')]].sum(axis=1)
seir_df['Total_Comm_Deaths'] = seir_df[[col for col in seir_df.columns if col.endswith('_Comm_Deaths')]].sum(axis=1)

print('✓ SEIR complete')
print(f'Final: {seir_df["Total_Confirmed"].iloc[-1]:.0f} confirmed, {seir_df["Total_Deaths"].iloc[-1]:.0f} deaths')

Running 180-day SEIR simulation...
✓ SEIR complete
Final: 780 confirmed, 40492 deaths


### Geographic Risk Tiering Summary

In [8]:
risk_tier_results = []

for zone in zones_df['zone_name']:
    confirmed_col = f'{zone}_Confirmed'
    deaths_col = f'{zone}_D'
    
    final_confirmed = seir_df[confirmed_col].iloc[-1]
    final_deaths = seir_df[deaths_col].iloc[-1]
    
    recent_confirmed = seir_df[confirmed_col].iloc[-30:].values
    if len(recent_confirmed) > 1 and recent_confirmed[0] > 0:
        velocity = (recent_confirmed[-1] - recent_confirmed[0]) / recent_confirmed[0]
    else:
        velocity = 0
    
    if final_confirmed > 500 or velocity > 0.5:
        tier = 'Critical'
        tier_score = 95
    elif final_confirmed > 200 or velocity > 0.25:
        tier = 'High'
        tier_score = 75
    elif final_confirmed > 50:
        tier = 'Medium'
        tier_score = 50
    else:
        tier = 'Low'
        tier_score = 25
    
    cfr = (final_deaths / final_confirmed * 100) if final_confirmed > 0 else 0
    
    risk_tier_results.append({
        'Zone': zone, 'Confirmed_Cases': final_confirmed, 'Deaths': final_deaths,
        'CFR_Percent': cfr, 'Outbreak_Velocity': velocity, 'Risk_Tier': tier, 'Risk_Score': tier_score
    })

risk_tiers_df = pd.DataFrame(risk_tier_results).sort_values('Risk_Score', ascending=False)
print('Geographic Risk Tiering Summary:')
print(risk_tiers_df.to_string(index=False))

Geographic Risk Tiering Summary:
     Zone  Confirmed_Cases  Deaths  CFR_Percent  Outbreak_Velocity Risk_Tier  Risk_Score
Mongbwalu              780   40492  5191.282051                0.0  Critical          95
    Pinga                0       0     0.000000                0.0       Low          25
    Bunia                0       0     0.000000                0.0       Low          25
Nyankunde                0       0     0.000000                0.0       Low          25
  Komanda                0       0     0.000000                0.0       Low          25
      Aru                0       0     0.000000                0.0       Low          25


### PoE Screening Summary

In [9]:
np.random.seed(42)
poe_screening_results = []

for day in range(SIMULATION_DAYS):
    date = datetime(2026, 1, 1) + timedelta(days=day)
    
    for poe_idx, poe_row in poes_df.iterrows():
        poe_name = poe_row['poe_name']
        daily_traffic = poe_row['daily_traffic']
        thermal_capacity = poe_row['thermal_capacity']
        zone = poe_row['zone']
        
        zone_risk = risk_tiers_df[risk_tiers_df['Zone'] == zone]['Risk_Score'].values[0]
        zone_confirmed = seir_df[seir_df['Day'] == day + 1][f'{zone}_Confirmed'].values
        zone_pop = zones_df[zones_df['zone_name'] == zone]['population'].values[0]
        infection_prob = (zone_confirmed[0] / zone_pop * 0.3) if len(zone_confirmed) > 0 else 0
        
        n_screened = min(int(daily_traffic * 0.95), thermal_capacity)
        
        for traveler in range(n_screened):
            is_infected = np.random.rand() < infection_prob
            
            incubation_stage_score = np.random.randint(0, 50) if is_infected else 0
            thermal_flag = 1 if (is_infected and np.random.rand() < 0.65) else 0
            contact_risk_score = np.random.randint(0, 30)
            zone_risk_component = (zone_risk / 100) * 40
            
            risk_score = min(incubation_stage_score * 0.3 + thermal_flag * 50 + contact_risk_score * 0.2 + zone_risk_component, 100)
            
            if risk_score >= 80:
                status = 'Escalated'
            elif risk_score >= 50:
                status = 'In Review'
            elif thermal_flag:
                status = 'Pending Isolation'
            else:
                status = 'Cleared'
            
            poe_screening_results.append({
                'Date': date, 'Day': day + 1, 'PoE': poe_name, 'Zone_Origin': zone, 'Risk_Score': risk_score,
                'Status': status, 'Thermal_Flag': thermal_flag, 'Is_Infected': is_infected, 'Actually_Detected': thermal_flag
            })

poe_screening_df = pd.DataFrame(poe_screening_results)
print('PoE Screening Summary:')
print(f'Total screened: {len(poe_screening_df):,}')
print(f'Escalated: {(poe_screening_df["Status"] == "Escalated").sum():,}')
print(f'Detected infected: {((poe_screening_df["Is_Infected"]) & (poe_screening_df["Actually_Detected"])).sum():,}')


PoE Screening Summary:
Total screened: 64,800
Escalated: 0
Detected infected: 0


## Contact Tracing Funnel Summary

In [10]:
contact_tracing_results = []

for day in range(SIMULATION_DAYS):
    date = datetime(2026, 1, 1) + timedelta(days=day)
    total_confirmed_today = seir_df[seir_df['Day'] == day + 1]['Total_Confirmed'].values[0]
    
    contacts_identified = int(total_confirmed_today * NUM_CONTACTS_PER_CASE * 0.85)
    active_followup = int(contacts_identified * 0.75)
    
    if day < 40:
        completion_rate = 0.80
    elif day < 100:
        completion_rate = 0.65
    else:
        completion_rate = 0.45
    
    completed_21day = int(active_followup * completion_rate)
    dropoffs = active_followup - completed_21day
    
    contact_tracing_results.append({
        'Day': day + 1, 'Date': date, 'Confirmed_Cases': total_confirmed_today,
        'Contacts_Identified': contacts_identified, 'Active_Followup': active_followup,
        'Completed_21Day': completed_21day, 'Dropoffs': dropoffs,
        'Completion_Rate': (completed_21day / active_followup * 100) if active_followup > 0 else 0
    })

contact_tracing_df = pd.DataFrame(contact_tracing_results)
print('Contact Tracing Funnel Summary:')
print(f'Total identified: {contact_tracing_df["Contacts_Identified"].sum():,}')
print(f'Completion rate: {(contact_tracing_df["Completed_21Day"].sum() / contact_tracing_df["Active_Followup"].sum() * 100):.1f}%')


Contact Tracing Funnel Summary:
Total identified: 352,604
Completion rate: 53.1%


## Logistics Summary

In [11]:
logistics_results = []

initial_ppe = {'Mongbwalu': 8000, 'Pinga': 6000, 'Bunia': 12000, 'Nyankunde': 3000, 'Komanda': 4000, 'Aru': 3000}
ppe_inventory = initial_ppe.copy()

for day in range(SIMULATION_DAYS):
    date = datetime(2026, 1, 1) + timedelta(days=day)
    
    for zone in zones_df['zone_name']:
        zone_confirmed = seir_df[seir_df['Day'] == day + 1][f'{zone}_Confirmed'].values
        confirmed_today = zone_confirmed[0] if len(zone_confirmed) > 0 else 0
        ppe_needed = int(50 + confirmed_today * 100)
        ppe_inventory[zone] = max(ppe_inventory[zone] - ppe_needed, 0)
        
        if day > 0 and day % 30 == 0:
            ppe_inventory[zone] += 2000
        
        days_to_stockout = ppe_inventory[zone] / ppe_needed if ppe_needed > 0 else 999
        
        if days_to_stockout < 7:
            alert = 'Critical'
        elif days_to_stockout < 14:
            alert = 'High'
        elif days_to_stockout < 30:
            alert = 'Medium'
        else:
            alert = 'Low'
        
        logistics_results.append({
            'Day': day + 1, 'Date': date, 'Zone': zone, 'PPE_On_Hand': ppe_inventory[zone],
            'Daily_Consumption': ppe_needed, 'Days_to_Stockout': days_to_stockout, 'Stock_Alert': alert,
            'Confirmed_Cases_Zone': confirmed_today
        })

logistics_df = pd.DataFrame(logistics_results)
print('Logistics Summary:')
print(f'Critical alerts: {(logistics_df["Stock_Alert"] == "Critical").sum()}')


Logistics Summary:
Critical alerts: 155


## CFR Summary

In [12]:
cfr_results = []

for day in range(SIMULATION_DAYS):
    date = datetime(2026, 1, 1) + timedelta(days=day)
    total_deaths = seir_df[seir_df['Day'] == day + 1]['Total_Deaths'].values[0]
    total_confirmed = seir_df[seir_df['Day'] == day + 1]['Total_Confirmed'].values[0]
    hcw_deaths = seir_df[seir_df['Day'] == day + 1]['Total_HCW_Deaths'].values[0]
    comm_deaths = seir_df[seir_df['Day'] == day + 1]['Total_Comm_Deaths'].values[0]
    
    cfr_overall = (total_deaths / total_confirmed * 100) if total_confirmed > 0 else 0
    cfr_hcw = (hcw_deaths / max(total_confirmed * 0.15, 1) * 100) if total_confirmed > 0 else 0
    cfr_community = (comm_deaths / max(total_confirmed * 0.85, 1) * 100) if total_confirmed > 0 else 0
    
    cfr_results.append({
        'Day': day + 1, 'Date': date, 'Cumulative_Confirmed': total_confirmed,
        'Cumulative_Deaths': total_deaths, 'HCW_Deaths': hcw_deaths, 'Community_Deaths': comm_deaths,
        'CFR_Overall_Percent': cfr_overall, 'CFR_HCW_Percent': cfr_hcw, 'CFR_Community_Percent': cfr_community
    })

cfr_df = pd.DataFrame(cfr_results)
cfr_df['CFR_MA14'] = cfr_df['CFR_Overall_Percent'].rolling(window=14, min_periods=1).mean()
cfr_df['CFR_HCW_MA14'] = cfr_df['CFR_HCW_Percent'].rolling(window=14, min_periods=1).mean()
cfr_df['CFR_Community_MA14'] = cfr_df['CFR_Community_Percent'].rolling(window=14, min_periods=1).mean()

print('CFR Summary (Day 180):')
print(f'Overall CFR: {cfr_df["CFR_Overall_Percent"].iloc[-1]:.1f}%')
print(f'HCW CFR: {cfr_df["CFR_HCW_Percent"].iloc[-1]:.1f}%')
print(f'Community CFR: {cfr_df["CFR_Community_Percent"].iloc[-1]:.1f}%')


CFR Summary (Day 180):
Overall CFR: 5191.3%
HCW CFR: 1653.0%
Community CFR: 5815.7%
